# Scorecard Feature Selection

Derives two feature sets from the same reproducible pipeline:

- `SCORECARD_FEATURES` - full candidate pool, keeps LendingClub's grade/rate.
- `APPLICATION_FEATURES` - excludes them. This is the set whose measured performance
  actually transfers to production, since a lender scoring its own applicants has no
  counterpart to `sub_grade` or `int_rate`.

Rerun whenever the target definition, the train window, or the WOE binning rules change.
All three changed since the last run, so the previous output is void.


In [ ]:
from pathlib import Path

import polars as pl
from sklearn.linear_model import LogisticRegression

from credit_risk.data.ingestion import load_raw_accepted_loans
from credit_risk.data.target import build_target
from credit_risk.evaluation.diagnostics import (
    coefficient_sign_report,
    drop_until_signs_are_clean,
    multicollinearity_report,
)
from credit_risk.features.build_dataset import (
    application_features,
    assemble_feature_matrix,
    gbm_features,
)
from credit_risk.features.woe import WOEEncoder, prune_correlated_features, rank_features_by_iv

pl.Config.set_tbl_rows(90)

CONFIG_PATH = Path("../configs/base.yaml")
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")


In [ ]:
df = load_raw_accepted_loans(DATA_PATH)
labeled = build_target(df, CONFIG_PATH)
final = assemble_feature_matrix(labeled, CONFIG_PATH)
train = final.filter(pl.col("split") == "train")

print(train.shape, "default_rate", round(train["default_flag"].mean(), 4))


## 1. Candidate pools


In [ ]:
full_pool = gbm_features(final)
app_pool = application_features(final)
print(f"full: {len(full_pool)}  application-only: {len(app_pool)}")
print("held out of app pool:", sorted(set(full_pool) - set(app_pool)))


## 2. IV ranking

`n_bins` is now the number of bins BEFORE monotonic merging (default 20), not the
final count. Watch the `strength` column: anything above 0.5 is a leakage alarm, not
a good feature. `sub_grade` sitting near the top is expected and is exactly why the
application-only pool exists.


In [ ]:
iv_full = rank_features_by_iv(train, full_pool)
print(iv_full.to_pandas().to_string(index=False))

iv_full.write_csv("../docs/iv_ranking_full.csv")


In [ ]:
above_threshold = iv_full.filter(pl.col("iv") >= 0.02)["feature"].to_list()
print(f"{len(above_threshold)} features with IV >= 0.02")


## 3. Binning audit

The artefact that makes binning decisions reviewable rather than implicit.

- `n_bins == 1`: the feature carries no usable signal after merging - drop it.
- `is_monotone` False on a NUMERIC feature: should not happen; investigate if it does.
  False on a categorical feature is expected, since categories have no ordering.
- `max_abs_woe` far above ~1.5: a thin bin is dominating; check `min_bin_bads`.


In [ ]:
encoder = WOEEncoder(features=above_threshold).fit(train)
report = encoder.binning_report()
print(report.to_pandas().to_string(index=False))

report.write_csv("../docs/binning_report.csv")
degenerate = report.filter(pl.col("n_bins") <= 1)["feature"].to_list()
print("single-bin features to drop:", degenerate)


## 4. Preliminary fit and diagnostics

This model is thrown away - it exists only to expose sign and collinearity problems.
All coefficients should be negative: WOE = ln(good/bad), so higher WOE means safer,
and a model predicting P(default=1) must weight it negatively.


In [ ]:
candidates = [f for f in above_threshold if f not in degenerate]
encoder = WOEEncoder(features=candidates).fit(train)
train_woe = encoder.transform(train)
woe_cols = [f"{f}_woe" for f in candidates]

prelim_model = LogisticRegression(max_iter=1000).fit(
    train_woe.select(woe_cols).to_pandas(), train_woe["default_flag"].to_pandas()
)

print(coefficient_sign_report(prelim_model, candidates).to_string(index=False))

collinearity = multicollinearity_report(train_woe, candidates, threshold=0.6)
print(collinearity.to_string(index=False) if len(collinearity) else "no pair above 0.6")


## 5. Prune correlated, then refine on signs


In [ ]:
def select_features(pool: list[str], train_df: pl.DataFrame) -> list[str]:
    """IV screen -> pairwise correlation pruning -> iterative sign-based refinement."""
    ranked = rank_features_by_iv(train_df, pool)
    kept = ranked.filter(pl.col("iv") >= 0.02)["feature"].to_list()

    enc = WOEEncoder(features=kept).fit(train_df)
    kept = [f for f in kept if enc.binning_report().filter(pl.col("feature") == f)["n_bins"][0] > 1]

    woe = WOEEncoder(features=kept).fit(train_df).transform(train_df)
    corr = woe.select([f"{f}_woe" for f in kept]).to_pandas()
    corr.columns = [c.replace("_woe", "") for c in corr.columns]
    pruned = prune_correlated_features(kept, corr.corr(), threshold=0.6)

    clean, _, model = drop_until_signs_are_clean(pruned, train_df)
    print(f"  {len(pool)} pool -> {len(kept)} IV -> {len(pruned)} pruned -> {len(clean)} final")
    print(coefficient_sign_report(model, clean).to_string(index=False))
    return clean

print("SCORECARD_FEATURES (full pool)")
scorecard_features = select_features(full_pool, train)
print(scorecard_features)


## 6. Application-only feature set

Same pipeline, `grade`/`sub_grade`/`int_rate`/`installment` removed. Expect a lower IV
ceiling and more features surviving the correlation prune, since `sub_grade` was
absorbing signal that other features also carry.


In [ ]:
print("APPLICATION_FEATURES (no lender-derived columns)")
application_selected = select_features(app_pool, train)
print(application_selected)


## 7. Output

Paste both lists into `features/build_dataset.py`, replacing the existing
`SCORECARD_FEATURES` and adding `APPLICATION_FEATURES`. Production code consumes the
decision; it must never re-derive it at runtime.


In [ ]:
print("SCORECARD_FEATURES = [")
print("    " + ", ".join(f'"{f}"' for f in scorecard_features))
print("]")
print()
print("APPLICATION_FEATURES = [")
print("    " + ", ".join(f'"{f}"' for f in application_selected))
print("]")
